In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

[ RAG 구현 절차 ]
1.	문서의 내용을 읽는다(document_loader를 이용)
- (1)	https://python.langchain.com/v0.2/docs/integrations/document_loaders/ 
- (2)	https://python.langchain.com/v0.2/docs/integrations/document_loaders/microsoft_word/
- %pip install --upgrade --quiet  docx2txt
2.	문서를 쪼갠다(한번에 이해하고 처리할 수 있는 입력+출력 토큰수가 제한)
- (1)	 https://python.langchain.com/v0.2/docs/how_to/recursive_text_splitter/#splitting-text-from-languages-without-word-boundaries 
- %pip install -qU langchain-text-splitters
3.	쪼갠 문서를 임베딩하여 vector database에 넣음
- (1)	OpenAIEmbeddings나 UpstageEmbeddings이용해서 임베딩
- (2)	https://python.langchain.com/v0.2/docs/integrations/vectorstores/chroma/  
- %pip install –q langchain-chroma
4.	질문을 이용해 유사도 검색
5.	유사도 검색한 문서를 LLM에 질문으로 전달하여 답변 얻음(제공되는 Prompt활용)
- (1)	https://python.langchain.com/v0.2/docs/tutorials/rag/
- %pip install –q langchain langchainhub
- http://smith.langchain.com에서 key생성 .env key 추가

# 2. 문서를 쪼개면서 읽기(O)

In [ ]:
import time
start = time.time()
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader('./tax_docs/소득세법(법률)(제20615호)(20250701).docx')
text_splitter = RecursiveCharacterTextSplitter( # 문서를 쪼개는 기준이 문자수
    chunk_size=1500, # 문서를 쪼갤때 1500글자씩 쪼개
    chunk_overlap=200
)
# 1번째 chunk 1~1450글자
# 2번째 chunk 1250~2790글자
documents = loader.load_and_split(text_splitter=text_splitter)
runtime = time.time() - start
print('문서 쪼개면서 읽는 시간 :', runtime)

# 3. 쪼갠문서를 임베딩 -> 벡터 데이터베이스 저장
- 임베딩 모델: upstage의 solar-embedding-1-large
- 벡터 데이터베이스: chroma
- 임베딩 객체 관련: https://python.langchain.com/v0.2/docs/integrations/text_embedding/upstage/#usage

In [2]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()

embeddings = UpstageEmbeddings(
    model="solar-embedding-1-large"
)

In [ ]:
doc_result = embeddings.embed_documents(
    ["소득세법 어쩌구 저쩌구", documents[0].page_content]
)
print(len(doc_result), len(doc_result[0]), len(doc_result[1]))

In [3]:
%%time
from langchain_chroma import Chroma
# 데이터를 처음 저장할 때만 실행, 반복실행시 누적되는 문제 있음
# database = Chroma.from_documents(
#     documents=documents,
#     embedding=embeddings,
#     collection_name='tax-collection', # 생략시 이름 랜덤
#     persist_directory='./chroma_upstage'      # 생략시 로컬데이터베이스에 저장안됨. 프로그램 종료시 db날라감
# )
database = Chroma(
    embedding_function=embeddings,
    collection_name='tax-collection',
    persist_directory='./chroma_upstage'
)

CPU times: total: 656 ms
Wall time: 698 ms


# 4. vector DB에 질문과 유사도 검색(답변 생성을 위한 retrieval)

In [4]:
query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'
retrieved_docs = database.similarity_search(query,
                                           k=3) # 기본 k값이 4

In [ ]:
retrieved_docs[0]

# 5. 유사도 검색으로 가져온 문서를 질문과 같이 LLM 전달하여 답변 생성

In [5]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4.1-nano')

In [6]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세 전문가입니다.
- [context]를 참고해서 사용자의 질문에 답변해 주세요.
- [context]는 다음과 같습니다.
{retrieved_docs}
Question : {query}"""

In [7]:
ai_message = llm.invoke(prompt)

In [ ]:
ai_message

In [8]:
print(ai_message.content)

연봉이 5천만원인 경우, 소득세 계산을 위해 우선 총급여액에 대한 각 공제액을 적용한 후 과세표준과 세율을 이용하여 세액을 산출해야 합니다. 다음은 일반적인 계산 과정입니다.

1. 근로소득공제 계산  
- 총급여액: 50,000,000원  
- 근로소득공제(최대 20,000,000원, 해당 금액 초과 시 20,000,000원 적용)  
→ 근로소득공제액: 20,000,000원

2. 과세표준 산출  
- 총급여액: 50,000,000원  
- 근로소득공제: 20,000,000원  
- 과세표준 = 총급여액 – 근로소득공제 = 30,000,000원

3. 종합소득세율 적용 (2023년 기준 표준 세율 참고)  
- 과세표준 30,000,000원에 대한 세율:  
→ 12% 구간 (1,200만원 초과 4,600만원 이하)  
→ 세액 = (과세표준 – 1,200만원) × 15% + 정액 (137만원)  
- 계산:  
→ (30,000,000 – 12,000,000) = 18,000,000  
→ 18,000,000 × 15% = 2,700,000  
→ 세액 = 1,370,000 + 2,700,000 = 4,070,000원

4. 기타 공제  
- 근로소득세액공제(최대 740,000원, 연 730만원 이상 과세표준 기준으로 약 74만원)  
→ 세액공제 후 최종 세액:  
→ 4,070,000원 – 740,000원 = 3,330,000원

따라서, 대략적인 소득세는 약 **3,330,000원**입니다.

**참고:** 재적공제, 자녀세액공제, 보험료공제 등 추가 공제 항목에 따라 최종 세액은 달라질 수 있으니 상세 계산을 원하시면 구체적 공제 내역을 알려주시기 바랍니다.


# 5. Augmentation을 위한 제공되는 prompt활용하여 langchain으로 답변 생성

In [9]:
query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'

from langchain import hub
prompt = hub.pull("rlm/rag-prompt")
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])

### RetrievalQA를 통해 LLM전달 (create_retrieval_chain이 대체)
```
query -> retriever전달(벡터 검색 수행) -> retrieval문서 -> prompt의 {context}에 삽입
    -> query -> prompt의 {question}에 삽입
```

In [10]:
from langchain.chains import RetrievalQA
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=database.as_retriever(search_kwargs={'k':3}),
    chain_type_kwargs={'prompt':prompt}
)

In [12]:
ai_message = qa_chain({"query": query})
print(ai_message)

{'query': '연봉 5천만원인 직장인의 소득세는 얼마인가요?', 'result': '연봉 5천만원인 직장인의 소득세는 구체적인 계산이 필요하지만, 일반적으로 근로소득공제와 자녀세액공제, 그리고 여러 공제 항목에 따라 결정됩니다. 근로소득공제는 최대 2천만원까지 적용되며, 자녀 1명인 경우 연 25만원, 2명인 경우 연 55만원, 3명 이상인 경우 55만원과 초과하는 자녀당 40만원이 공제됩니다. 정확한 세액은 상세한 소득액과 공제 대상 여부에 따라 달라지므로, 세무 전문가와 상담하는 것이 좋습니다.'}
